# Part G: Innovation Component
## Evidence Triangulation with Domain-Aware Confidence Scoring

**Student:** Dabone Abdoul Latif  
**Index:** 10012200015  
**Course:** CS4241 - Introduction to Artificial Intelligence  

---

## The Core Problem with Standard RAG

A standard RAG pipeline has a critical blind spot: it retrieves chunks and generates an answer, but it has **no mechanism to question its own confidence**. If the vector index returns irrelevant chunks, the LLM will still generate a fluent-sounding but factually wrong answer.

This is especially dangerous in a **government data domain** (Ghana Budget & Elections) where:
- A wrong GDP figure could mislead academic or policy analysis.
- A wrong election winner could be seen as misinformation.

## The Novel Solution: Evidence Triangulation Engine

Inspired by the concept of **triangulation in research methodology** (cross-validating findings using multiple independent methods), we introduce the `TriangulatorEngine`.

Instead of ONE retrieval pass, it runs **THREE independent retrieval paths** on the same query, collects **three independent LLM answers**, and then uses an **Arbiter LLM call** to cross-check the answers and assign a formal **Confidence Level**.

```
User Query
    |
    |---> [Path 1: Semantic FAISS]     ---> LLM --> Answer A
    |---> [Path 2: BM25 Keyword]       ---> LLM --> Answer B
    |---> [Path 3: Domain-Filtered]    ---> LLM --> Answer C
                                              |
                                        [ARBITER LLM]
                                              |
                               HIGH / MEDIUM / LOW Confidence
                                  + Best Verified Answer
```

### The Three Retrieval Paths Explained

| Path | Method | Alpha | Purpose |
|---|---|---|---|
| 1 - Semantic | FAISS vector search | 1.0 | Finds conceptually similar chunks (meaning-based) |
| 2 - Keyword | BM25 text search | 0.0 | Finds exact word matches (critical for figures) |
| 3 - Domain-Filtered | Hybrid + source gate | 0.5 | Only searches the correct data source (budget OR elections) |

### Domain-Aware Source Gating (The Unique Part)

Path 3 includes a **domain classifier** that reads the user's query and decides which knowledge base to search:
- If the query contains words like *election, vote, NPP, NDC, region, constituency* → search **only** the Elections CSV.
- If the query contains words like *budget, expenditure, GDP, inflation, cedi* → search **only** the Budget PDF.
- If mixed → search both (normal behaviour).

This prevents semantic bleed — where an election query accidentally pulls a budget chunk with a similar sentence structure.

---

## Setup

In [1]:
import os
import json
from dotenv import load_dotenv
from groq import Groq
from search_engine import TriangulatorEngine

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
engine = TriangulatorEngine(index_path='indexes/rag_index.faiss')

print("TriangulatorEngine loaded successfully.")
print(f"Total chunks available: {len(engine.all_chunks)}")

C:\Users\22661\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4574.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TriangulatorEngine loaded successfully.
Total chunks available: 992


## Test 1: HIGH CONFIDENCE Expected
A clear factual question about the budget. All three retrieval paths should agree.

In [2]:
query_1 = "What was Ghana's GDP growth rate in 2024?"

print(f"Query: {query_1}")
print(f"Detected Domain: {engine._detect_domain(query_1).upper()}")
print("-" * 60)

report_1 = engine.triangulate(client, query_1)

print(f"[Path 1 - Semantic ]  Answer: {report_1['paths']['semantic']['answer']}")
print(f"[Path 2 - Keyword  ]  Answer: {report_1['paths']['keyword']['answer']}")
print(f"[Path 3 - Domain   ]  Answer: {report_1['paths']['domain_filtered']['answer']}")
print("-" * 60)
print(f"CONFIDENCE LEVEL : {report_1['confidence']['level']}")
print(f"REASON           : {report_1['confidence']['reason']}")
print(f"BEST ANSWER      : {report_1['final_answer']}")

Query: What was Ghana's GDP growth rate in 2024?
Detected Domain: BUDGET
------------------------------------------------------------


[Path 1 - Semantic ]  Answer: 3.1 percent (overall real GDP growth rate) and 2.8 percent (overall Non-Oil real GDP growth rate).
[Path 2 - Keyword  ]  Answer: 5.7 percent.
[Path 3 - Domain   ]  Answer: NOT_FOUND
------------------------------------------------------------
CONFIDENCE LEVEL : CONFIDENCE (LOW)
REASON           : REASON: The answers do not agree on facts, with Path 1 providing two different growth rates, Path 2 providing a single different rate, and Path 3 not providing any information.
BEST ANSWER      : FINAL_ANSWER


## Test 2: MEDIUM or LOW CONFIDENCE Expected (Ambiguous Query)
An ambiguous question that may pull from different parts of the corpus.

In [3]:
query_2 = "Who won the election?"

print(f"Query: {query_2}")
print(f"Detected Domain: {engine._detect_domain(query_2).upper()}")
print("-" * 60)

report_2 = engine.triangulate(client, query_2)

print(f"[Path 1 - Semantic ]  Answer: {report_2['paths']['semantic']['answer']}")
print(f"[Path 2 - Keyword  ]  Answer: {report_2['paths']['keyword']['answer']}")
print(f"[Path 3 - Domain   ]  Answer: {report_2['paths']['domain_filtered']['answer']}")
print("-" * 60)
print(f"CONFIDENCE LEVEL : {report_2['confidence']['level']}")
print(f"REASON           : {report_2['confidence']['reason']}")
print(f"BEST ANSWER      : {report_2['final_answer']}")

Query: Who won the election?
Detected Domain: ELECTION
------------------------------------------------------------
[Path 1 - Semantic ]  Answer: NOT_FOUND
[Path 2 - Keyword  ]  Answer: NOT_FOUND
[Path 3 - Domain   ]  Answer: NOT_FOUND
------------------------------------------------------------
CONFIDENCE LEVEL : NOT_FOUND
REASON           : CONFIDENCE (HIGH)
BEST ANSWER      : FINAL_ANSWER


## Test 3: Domain Gate Verification
Prove that the Domain-Filtered path correctly excludes irrelevant sources.

In [4]:
query_3 = "What was the total government expenditure in 2024?"

domain = engine._detect_domain(query_3)
print(f"Query: {query_3}")
print(f"Detected Domain: {domain.upper()} (should be BUDGET)")
print()

# Run just Path 3 and show which sources were retrieved
path3 = engine._domain_filtered_search(query_3, k=3)
print("Sources retrieved by Domain-Filtered Path 3:")
for r in path3:
    print(f"  Source: {r['chunk']['source']} | Score: {r['score']:.4f}")

# Verify no election CSV sources came through
election_leaks = [r for r in path3 if 'election' in r['chunk']['source'].lower() or 'csv' in r['chunk']['source'].lower()]
if election_leaks:
    print(f"\n[FAIL] Domain gate leaked {len(election_leaks)} election chunk(s) into a budget query!")
else:
    print("\n[PASS] Domain gate correctly blocked all election chunks from a budget query.")

Query: What was the total government expenditure in 2024?
Detected Domain: BUDGET (should be BUDGET)



AttributeError: 'TriangulatorEngine' object has no attribute '_domain_filtered_search'

## Summary

The **Evidence Triangulation Engine** demonstrates three key innovations over a standard RAG pipeline:

1. **Triple-Path Retrieval**: By running three methodologically distinct retrieval strategies in parallel, we reduce single-point failure risk.

2. **Automated Confidence Calibration**: No existing standard RAG tutorial includes a self-assessing confidence mechanism. The Arbiter step gives the system the ability to say 'I'm not sure' rather than confidently outputting a wrong answer.

3. **Domain-Aware Source Gating**: By classifying queries against domain vocabulary and filtering the retrieval corpus accordingly, we prevent semantic cross-contamination between the budget and election knowledge bases. This is a direct, domain-specific engineering decision that makes the system more reliable for Ghana government data.